# BIRD Rewriter Ablation — Graph + DFC + Alias Norm + Column Repair + Execute-and-Retry

**Component 3** — the full reactive layer. Layered on top of [graph](graph.md). Same upstream pipeline (Qwen-7B expansion → BM25/dense+RRF retrieval at N=8 → minimal connecting subgraph → Qwen-32B SQL gen). The new piece: a four-stage reactive layer that processes the model's SQL output:

1. **Schema-grounded column repair** — sqlglot AST walk; for each column reference, verify the column exists in its referenced table. If not, fuzzy-match within the table; if still no match, rewrite to a sibling linked table. Catches `qid=15`-style hallucinated columns.
2. **DFC rewriter** — project's `sql_rewriter` module (DuckDB-backed AST transformer) enforces FK constraints as policy rules.
3. **Alias normalizer** — sqlglot pass that rewrites column refs from real table names to declared aliases (`frpm.CDSCode` → `T1.CDSCode` when `frpm AS T1`). Idempotent.
4. **Execute-and-retry** — execute the repaired SQL on the actual SQLite DB. On error, build a retry prompt with `(linked schema, evidence, question, bad SQL, error message)` and regenerate via Qwen-32B. One retry max.

```
graph join_plan + question + evidence
      ↓
[Qwen-32B] → SQL_v0
      ↓
[Column repair (AST)]    → SQL_v1
      ↓
[DFC + AST join repair]  → SQL_v2
      ↓
[Alias normalizer]       → SQL_v3
      ↓
[Execute on actual DB]
      ├─ no error → final = SQL_v3
      └─ error    → [Qwen-32B retry with error context] → SQL_retry
                    → column repair → alias norm → re-execute
                    → final = SQL_retry (or SQL_v3 if retry also errors)
```

**Comparison frame:**

| Method | Join Acc. | Exec. Acc. |
|---|---|---|
| Baseline (Qwen-32B, full schema) | 66.3 | 54.2 |
| Retrieval (N=5) | 66.7 | 52.1 |
| + Graph (N=8) | 71.9 | 53.3 |
| **+ Rewriter (this run)** | **?** | **?** |

**Hypothesis.** Column repair catches qid=15-style hallucinations directly. Execute-and-retry recovers most of the 89 errored predictions from the graph run. Together: EX 53.3 → 58–62, errors 89 → ~30.

Predictions: `/content/predict_dev_rewriter.json`. Per-question telemetry: `/content/rewriter_log.jsonl`.

## 1. Setup

In [ ]:
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/mayursk2000/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents.git"
REPO_DIR = pathlib.Path("/content/Stop-Guessing-Joins-Deterministic-Schema-Linking-for-Text-to-SQL-Agents")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "sqlglot", "huggingface_hub", "requests", "tqdm", "psutil",
    "transformers>=4.45", "accelerate>=0.34", "bitsandbytes>=0.43", "sentencepiece",
    "rank_bm25", "sentence-transformers",
    "duckdb", "botocore",  # required by DFC rewriter
])
print("Repo:", REPO_DIR)

## 2. Secrets — `HF_TOKEN`

In [ ]:
HF_TOKEN = None
try:
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not available:", e)
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

## 3. GPU check

In [ ]:
import torch

GEN_MODEL_NAME       = "unsloth/Qwen2.5-Coder-32B-Instruct-bnb-4bit"
EXPANSION_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
ENCODER_NAME         = "BAAI/bge-small-en-v1.5"

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name}  |  VRAM: {vram_gb:.1f} GB")
print("Generator:", GEN_MODEL_NAME)
print("Expansion:", EXPANSION_MODEL_NAME)
print("Encoder:  ", ENCODER_NAME)

## 4. Download BIRD dev

In [ ]:
import re, zipfile, json, urllib.request

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown

BIRD_ROOT = pathlib.Path("/content/bird")
BIRD_ROOT.mkdir(exist_ok=True)
BIRD_DEV_URL = "https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip"
zip_path = BIRD_ROOT / "dev.zip"

def _looks_like_html(p):
    if not p.exists() or p.stat().st_size < 1_000_000:
        try:
            head = p.read_bytes()[:512].lower()
            return b"<!doctype html" in head or b"<html" in head
        except Exception:
            return True
    return False

def _drive_id(url):
    for pat in (r"/file/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, url)
        if m: return m.group(1)
    return None

if zip_path.exists() and _looks_like_html(zip_path):
    zip_path.unlink()
if BIRD_DEV_URL and not zip_path.exists():
    drive_id = _drive_id(BIRD_DEV_URL)
    if drive_id:
        gdown.download(id=drive_id, output=str(zip_path), quiet=False)
    else:
        urllib.request.urlretrieve(BIRD_DEV_URL, zip_path)

assert zip_path.exists() and not _looks_like_html(zip_path)

if not list(BIRD_ROOT.rglob("dev.json")):
    print("Extracting ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(BIRD_ROOT)
    for inner in BIRD_ROOT.rglob("dev_databases.zip"):
        with zipfile.ZipFile(inner) as z:
            z.extractall(inner.parent)
        inner.unlink()
    zip_path.unlink()

candidates = [p.parent for p in BIRD_ROOT.rglob("dev.json")]
BIRD_DEV = candidates[0]
BIRD_DBS = next(BIRD_DEV.rglob("dev_databases"), BIRD_DEV / "dev_databases")
print("BIRD_DEV =", BIRD_DEV)
print("BIRD_DBS =", BIRD_DBS)
print("Questions:", len(json.loads((BIRD_DEV / "dev.json").read_text())))

## 5. Schema introspection

In [ ]:
import sqlite3, csv
from text2sql_agent_prototype.prototype import Schema, Table, Column, ForeignKey

_BIRD_TABLES_JSON = None
def _load_dev_tables():
    global _BIRD_TABLES_JSON
    if _BIRD_TABLES_JSON is None:
        path = next(BIRD_DEV.rglob("dev_tables.json"), None)
        _BIRD_TABLES_JSON = {b["db_id"]: b for b in json.loads(path.read_text())} if path else {}
    return _BIRD_TABLES_JSON

def _read_column_descriptions(db_id):
    desc_dir = BIRD_DBS / db_id / "database_description"
    out = {}
    if not desc_dir.exists():
        return out
    for csv_path in desc_dir.glob("*.csv"):
        tbl = csv_path.stem.lower()
        col_descs = {}
        try:
            with open(csv_path, encoding="utf-8", errors="replace") as f:
                for row in csv.DictReader(f):
                    col = (row.get("original_column_name") or "").strip()
                    desc = (row.get("column_description") or "").strip()
                    val_desc = (row.get("value_description") or "").strip()
                    full = " ; ".join(filter(None, [desc, val_desc]))
                    if col:
                        col_descs[col.lower()] = full
        except Exception as exc:
            print(f"  warn: {csv_path.name}: {exc}")
        out[tbl] = col_descs
    return out

_SCHEMA_CACHE = {}
def schema_from_bird(db_id):
    if db_id in _SCHEMA_CACHE:
        return _SCHEMA_CACHE[db_id]
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path))
    try:
        cur = conn.cursor()
        table_names = [r[0] for r in cur.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'"
        ).fetchall()]
        col_descs = _read_column_descriptions(db_id)
        bird_meta = _load_dev_tables().get(db_id)

        tables = {}
        for tname in table_names:
            cols_info = cur.execute(f'PRAGMA table_info("{tname}")').fetchall()
            tbl_descs = col_descs.get(tname.lower(), {})
            columns = tuple(
                Column(name=c[1], description=" | ".join(filter(None, [
                    c[2].lower() if c[2] else "",
                    tbl_descs.get(c[1].lower(), ""),
                ])))
                for c in cols_info
            )
            tbl_desc = tname.replace("_", " ")
            if bird_meta and tname in bird_meta.get("table_names_original", []):
                idx = bird_meta["table_names_original"].index(tname)
                if idx < len(bird_meta.get("table_names", [])):
                    tbl_desc = bird_meta["table_names"][idx] or tbl_desc
            tables[tname] = Table(name=tname, columns=columns, description=tbl_desc)

        foreign_keys = []
        if bird_meta:
            names_orig = bird_meta["table_names_original"]
            cols_orig  = bird_meta["column_names_original"]
            for fk_pair in bird_meta.get("foreign_keys", []):
                l_idx, r_idx = fk_pair[0], fk_pair[1]
                lt = names_orig[cols_orig[l_idx][0]]; lc = cols_orig[l_idx][1]
                rt = names_orig[cols_orig[r_idx][0]]; rc = cols_orig[r_idx][1]
                if lt in tables and rt in tables:
                    foreign_keys.append(ForeignKey(left_table=lt, left_column=lc,
                                                   right_table=rt, right_column=rc))
        seen = {(fk.left_table, fk.left_column, fk.right_table, fk.right_column) for fk in foreign_keys}
        for tname in table_names:
            for fk in cur.execute(f'PRAGMA foreign_key_list("{tname}")').fetchall():
                _, _, ref_table, from_col, to_col, *_ = fk
                if ref_table in tables and (tname, from_col, ref_table, to_col) not in seen:
                    foreign_keys.append(ForeignKey(left_table=tname, left_column=from_col,
                                                   right_table=ref_table, right_column=to_col))

        sch = Schema(tables=tables, foreign_keys=foreign_keys)
        _SCHEMA_CACHE[db_id] = sch
        return sch
    finally:
        conn.close()

def load_dev(bird_dev_dir):
    return json.loads((bird_dev_dir / "dev.json").read_text())

## 6. Sanity check

In [ ]:
questions = load_dev(BIRD_DEV)
print(f"Total questions: {len(questions)}")
sch0 = schema_from_bird(questions[0]["db_id"])
print(f"Sample db={questions[0]['db_id']}: {len(sch0.tables)} tables, {len(sch0.foreign_keys)} FKs")

## 7. Hybrid retriever (same as graph)

In [ ]:
import re as _re
from collections import defaultdict
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from text2sql_agent_prototype.prototype import HybridRetriever, RetrievalResult, RetrievalMatch

print(f"Loading encoder {ENCODER_NAME} ...")
encoder = SentenceTransformer(ENCODER_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
print("Encoder loaded.")

def _smart_split(text):
    out = []
    for word in _re.findall(r"[A-Za-z0-9]+", text):
        for piece in _re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|[0-9]+", word):
            t = piece.lower()
            if len(t) >= 2:
                out.append(t)
    return out

class BIRDHybridRetriever(HybridRetriever):
    RRF_K = 60
    def __init__(self, schema, db_id, top_k=12):
        self.schema = schema; self.db_id = db_id; self.top_k = top_k; self.min_score = 0.0
        self.docs = []; self.doc_tables = []
        for tname, table in schema.tables.items():
            for c in table.columns:
                self.docs.append(f"Table {tname} ({table.description}). Column {c.name}: {c.description or ''}")
                self.doc_tables.append(tname)
        self.bm25 = BM25Okapi([_smart_split(d) for d in self.docs])
        self.dense = encoder.encode(self.docs, normalize_embeddings=True,
                                    show_progress_bar=False, convert_to_numpy=True)

    def _rank_tables(self, query):
        if not self.docs: return []
        bm25_scores = self.bm25.get_scores(_smart_split(query))
        bm25_rank = {int(i): r for r, i in enumerate(np.argsort(-bm25_scores))}
        q_emb = encoder.encode([query], normalize_embeddings=True,
                               show_progress_bar=False, convert_to_numpy=True)[0]
        dense_scores = self.dense @ q_emb
        dense_rank = {int(i): r for r, i in enumerate(np.argsort(-dense_scores))}
        K = self.RRF_K
        rrf = {i: 1.0/(K+bm25_rank[i]) + 1.0/(K+dense_rank[i]) for i in range(len(self.docs))}
        table_score = {}
        for i, tname in enumerate(self.doc_tables):
            s = rrf[i]
            if tname not in table_score or s > table_score[tname]:
                table_score[tname] = s
        return sorted(table_score.items(), key=lambda kv: -kv[1])

    def retrieve(self, query):
        ranked = self._rank_tables(query)[:self.top_k]
        return RetrievalResult(query=query,
                               candidate_tables=[t for t, _ in ranked],
                               matches=[RetrievalMatch(table=t, lexical_score=0.0,
                                                       semantic_score=0.0, score=float(s),
                                                       matched_terms=[]) for t, s in ranked])

_RETRIEVER_CACHE = {}
def get_retriever(db_id):
    if db_id not in _RETRIEVER_CACHE:
        _RETRIEVER_CACHE[db_id] = BIRDHybridRetriever(schema_from_bird(db_id), db_id)
    return _RETRIEVER_CACHE[db_id]

## 8. Qwen-7B expansion (same as graph)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import re, gc, json as _json

print(f"Loading expansion model {EXPANSION_MODEL_NAME} (BF16) ...")
exp_tokenizer = AutoTokenizer.from_pretrained(EXPANSION_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
exp_tokenizer.padding_side = "left"; exp_tokenizer.truncation_side = "left"
if exp_tokenizer.pad_token_id is None:
    exp_tokenizer.pad_token = exp_tokenizer.eos_token
    exp_tokenizer.pad_token_id = exp_tokenizer.eos_token_id
exp_model = AutoModelForCausalLM.from_pretrained(
    EXPANSION_MODEL_NAME, token=HF_TOKEN,
    torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
exp_model.eval()
print("Expansion model loaded.")

EXPANSION_SYSTEM = (
    "You are a query-expansion assistant for a Text-to-SQL retrieval system. "
    "You receive a natural-language database question and optional evidence "
    "hints. Emit JSON with two fields:\n"
    '  "paraphrases": 3 alternate phrasings.\n'
    '  "entities": 1-8 noun phrases that look like tables/columns/values.\n\n'
    "Output ONLY the JSON object."
)
EXPANSION_FEWSHOT = [
    {"role": "user", "content":
        "Question: How many charter schools in Fresno County have free meal counts above 100?\n"
        "Evidence: Charter schools refers to `Charter School (Y/N)` = 1 in the frpm"},
    {"role": "assistant", "content":
        '{"paraphrases": ["Count charter schools located in Fresno County with '
        'free meal counts greater than 100", "How many institutions categorized '
        'as charter in Fresno County have more than 100 free meals", "Number '
        'of charter-status schools in the Fresno County area where free meal '
        'count exceeds 100"], "entities": ["charter schools", "Fresno County", '
        '"free meal count", "Charter School (Y/N)", "frpm"]}'},
]

_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def _parse_expansion(raw, fallback_question):
    m = _JSON_RE.search(raw)
    if not m: return {"paraphrases": [fallback_question], "entities": []}
    try: obj = _json.loads(m.group(0))
    except Exception: return {"paraphrases": [fallback_question], "entities": []}
    paras = obj.get("paraphrases") or []
    if not isinstance(paras, list): paras = [str(paras)]
    paras = [str(p).strip() for p in paras if str(p).strip()][:3] or [fallback_question]
    ents = obj.get("entities") or []
    if not isinstance(ents, list): ents = [str(ents)]
    ents = [str(e).strip() for e in ents if str(e).strip()][:8]
    return {"paraphrases": paras, "entities": ents}

def expand_query_batch(items, batch_size=16):
    if not items: return []
    results = []
    for start in range(0, len(items), batch_size):
        chunk = items[start:start + batch_size]
        prompts = []
        for q, ev in chunk:
            user = f"Question: {q}" + (f"\nEvidence: {ev}" if ev else "")
            msgs = [{"role": "system", "content": EXPANSION_SYSTEM},
                    *EXPANSION_FEWSHOT,
                    {"role": "user", "content": user}]
            prompts.append(exp_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        try:
            inputs = exp_tokenizer(prompts, return_tensors="pt", padding=True,
                                    truncation=True, max_length=2048).to(exp_model.device)
            with torch.inference_mode():
                out = exp_model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                         pad_token_id=exp_tokenizer.pad_token_id)
            decoded = exp_tokenizer.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
            del out, inputs
            try: torch.cuda.empty_cache()
            except Exception: pass
        except BaseException as exc:
            if isinstance(exc, (KeyboardInterrupt, SystemExit)): raise
            print(f"  expansion batch failed: {exc}")
            decoded = [""] * len(chunk)
        for raw, (q, _ev) in zip(decoded, chunk):
            results.append(_parse_expansion(raw, q))
    return results

print("expand_query_batch ready.")

## 9. Retrieval (N=8) + minimal connecting subgraph

In [ ]:
from text2sql_agent_prototype.prototype import SchemaGraph, JoinPlan

N_RETRIEVAL_TABLES = 8

_GRAPH_CACHE = {}
def get_graph(db_id):
    if db_id not in _GRAPH_CACHE:
        _GRAPH_CACHE[db_id] = SchemaGraph(schema_from_bird(db_id))
    return _GRAPH_CACHE[db_id]

def retrieve_then_graph(db_id, question, evidence, expansion, n_retrieval=N_RETRIEVAL_TABLES):
    retriever = get_retriever(db_id)
    variants = []
    anchor = (question + " " + evidence).strip() if evidence else question
    variants.append(anchor)
    variants.extend(expansion.get("paraphrases", []))
    variants.extend(expansion.get("entities", []))
    K = 60
    table_rrf = defaultdict(float)
    for v in variants:
        if not v.strip(): continue
        for rank, (tname, _s) in enumerate(retriever._rank_tables(v)):
            table_rrf[tname] += 1.0 / (K + rank)
    retrieved = [t for t, _ in sorted(table_rrf.items(), key=lambda kv: -kv[1])[:n_retrieval]]
    graph = get_graph(db_id)
    join_plan = graph.minimal_connecting_subgraph(retrieved)
    bridges = [t for t in join_plan.tables if t not in retrieved]
    return join_plan, {
        "n_variants": len(variants),
        "retrieved": retrieved,
        "graph_tables": list(join_plan.tables),
        "bridges": bridges,
        "unresolved": list(join_plan.unresolved_tables),
        "approved_joins": [j.join_condition() for j in join_plan.joins],
    }

## 10. Load Qwen-32B and define generators (initial + retry)

In [ ]:
from transformers import BitsAndBytesConfig
import re, gc

USE_4BIT          = True
MAX_NEW_TOKENS    = 256
MAX_INPUT_TOKENS  = 8192
MAX_DESC_LEN      = 100
MAX_TABLE_COLUMNS = 80

print(f"Loading {GEN_MODEL_NAME} (4-bit={USE_4BIT}) ...")
_bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME, token=HF_TOKEN, trust_remote_code=True)
gen_tokenizer.padding_side = "left"; gen_tokenizer.truncation_side = "left"
if gen_tokenizer.pad_token_id is None:
    gen_tokenizer.pad_token = gen_tokenizer.eos_token
    gen_tokenizer.pad_token_id = gen_tokenizer.eos_token_id
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME, token=HF_TOKEN, quantization_config=_bnb_cfg,
    torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True,
)
gen_model.eval()
print("Generator loaded.")

_FENCE = re.compile(r"^```(?:sql|sqlite)?\s*|\s*```$", re.IGNORECASE | re.MULTILINE)

GEN_SYSTEM_PROMPT = (
    "You are an expert Text-to-SQL assistant for SQLite, evaluated on BIRD. "
    "You receive: (1) a SCHEMA-LINKED set of tables (resolved by retrieval + "
    "FK graph traversal), (2) an APPROVED-JOINS list, (3) optional BIRD evidence, "
    "(4) a natural-language question.\n\n"
    "STRICT RULES:\n"
    "- If the evidence names a specific column in backticks, USE THAT EXACT COLUMN.\n"
    "- If the evidence provides a formula (e.g. \"rate = A / B\"), COMPUTE that "
    "expression literally; for ratios CAST the numerator to REAL.\n"
    "- If the evidence gives a literal value, use that EXACT value.\n"
    "- Use ONLY tables and columns that appear in the linked schema below.\n"
    "- When joining, you MUST use only predicates from the APPROVED-JOINS list.\n"
    "- The question may need only ONE table. Do not force a join unless required.\n"
    "- Quote column names with spaces or special characters with double quotes.\n"
    "- Return EXACTLY ONE SQLite SELECT statement. No prose, no code fences."
)

RETRY_SYSTEM_PROMPT = (
    "You are repairing a failed SQL query. The previous attempt threw a SQLite "
    "execution error. Read the error carefully, look at the linked schema and "
    "approved joins, and emit a corrected SELECT statement.\n\n"
    "STRICT RULES (same as before):\n"
    "- Use ONLY columns that appear in the linked schema below.\n"
    "- When joining, use ONLY predicates from the APPROVED-JOINS list.\n"
    "- Match exact column names from the evidence, including spaces/special characters.\n"
    "- Return EXACTLY ONE corrected SQLite SELECT statement. No prose."
)

def _trim(s):
    s = (s or "").replace("\r", " ").replace("\n", " ").replace("\t", " ").strip()
    if len(s) > MAX_DESC_LEN:
        s = s[:MAX_DESC_LEN].rstrip() + "…"
    return s

def linked_schema_block(db_id, join_plan):
    schema = schema_from_bird(db_id)
    blocks = []
    all_tables = list(dict.fromkeys([*join_plan.tables, *join_plan.unresolved_tables]))
    for tname in all_tables:
        if tname not in schema.tables: continue
        table = schema.tables[tname]
        lines = []
        for c in table.columns[:MAX_TABLE_COLUMNS]:
            desc = _trim(c.description or "")
            base = f'    "{c.name}"'
            lines.append(f"{base}  -- {desc}" if desc else base)
        if len(table.columns) > MAX_TABLE_COLUMNS:
            lines.append(f"    -- ... {len(table.columns) - MAX_TABLE_COLUMNS} more columns omitted")
        label = "" if tname in join_plan.tables else "  -- (unresolved candidate, no FK path)"
        blocks.append(f'CREATE TABLE "{tname}" ({label}\n' + ",\n".join(lines) + "\n);")
    return "\n\n".join(blocks)

def approved_joins_block(join_plan):
    if not join_plan.joins:
        return "(no joins required — single-table query is acceptable)"
    return "\n".join(f"  {j.join_condition()}" for j in join_plan.joins)

def _build_gen_messages(db_id, question, evidence, join_plan):
    user_parts = [
        f"Linked schema (graph-resolved from retrieved candidates):\n{linked_schema_block(db_id, join_plan)}",
        f"Approved joins (use ONLY these):\n{approved_joins_block(join_plan)}",
    ]
    if evidence:
        user_parts.append(f"BIRD evidence — REQUIRED column hints:\n{evidence}")
    user_parts.append(f"Question:\n{question}")
    user_parts.append("Output only the SELECT statement.")
    return [{"role": "system", "content": GEN_SYSTEM_PROMPT},
            {"role": "user",   "content": "\n\n".join(user_parts)}]

def _build_retry_messages(db_id, question, evidence, join_plan, bad_sql, error_msg):
    user_parts = [
        f"Linked schema:\n{linked_schema_block(db_id, join_plan)}",
        f"Approved joins:\n{approved_joins_block(join_plan)}",
    ]
    if evidence:
        user_parts.append(f"BIRD evidence:\n{evidence}")
    user_parts.append(f"Question:\n{question}")
    user_parts.append(f"Previous SQL attempt (FAILED with: {error_msg[:300]}):\n{bad_sql}")
    user_parts.append("Emit the corrected SELECT statement. Output only SQL.")
    return [{"role": "system", "content": RETRY_SYSTEM_PROMPT},
            {"role": "user",   "content": "\n\n".join(user_parts)}]

def _clean_sql(text):
    text = _FENCE.sub("", text).strip()
    m = re.search(r"\b(SELECT|WITH)\b", text, re.IGNORECASE)
    if m: text = text[m.start():]
    return text.strip().rstrip(";").strip()

def _validate_sql(sql):
    if not re.match(r"(?i)^\s*(select|with)\b", sql):
        raise ValueError(f"Non-SELECT output: {sql[:120]}")
    return sql

def _free_cache():
    try: torch.cuda.empty_cache()
    except Exception: pass

def generate_sql_one(db_id, question, evidence, join_plan):
    msgs = _build_gen_messages(db_id, question, evidence, join_plan)
    prompt = gen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(gen_model.device)
    with torch.inference_mode():
        out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=gen_tokenizer.pad_token_id)
    raw = gen_tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    del out, inputs; _free_cache()
    return _validate_sql(_clean_sql(raw))

def generate_sql_batch(items, max_input_length=MAX_INPUT_TOKENS):
    if not items: return []
    out = inputs = new_token_rows = None
    try:
        prompts = [gen_tokenizer.apply_chat_template(_build_gen_messages(db, q, ev, jp),
                                                      tokenize=False, add_generation_prompt=True)
                   for (db, q, ev, jp) in items]
        prompt_lengths = [len(gen_tokenizer(p, add_special_tokens=False).input_ids) for p in prompts]
        if prompt_lengths and max(prompt_lengths) > max_input_length:
            print(f"  truncating gen batch: max_tokens={max(prompt_lengths)} cap={max_input_length}")
        inputs = gen_tokenizer(prompts, return_tensors="pt", padding=True,
                               truncation=True, max_length=max_input_length).to(gen_model.device)
        with torch.inference_mode():
            out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                     pad_token_id=gen_tokenizer.pad_token_id)
        new_token_rows = out[:, inputs.input_ids.shape[1]:]
        decoded = gen_tokenizer.batch_decode(new_token_rows, skip_special_tokens=True)
    except BaseException as exc:
        if isinstance(exc, (KeyboardInterrupt, SystemExit)): raise
        print(f"  gen batch failed: {exc}; batch={len(items)}")
        del out, inputs; _free_cache(); gc.collect()
        if len(items) > 1:
            mid = len(items) // 2
            return generate_sql_batch(items[:mid], max_input_length) + generate_sql_batch(items[mid:], max_input_length)
        try: return [generate_sql_one(*items[0])]
        except Exception: return ["SELECT 1"]
    finally:
        try: del out, inputs, new_token_rows
        except NameError: pass
        _free_cache()
    return [
        (lambda raw: _validate_sql(_clean_sql(raw)) if re.search(r"\b(SELECT|WITH)\b", raw, re.IGNORECASE) else "SELECT 1")(r)
        if r else "SELECT 1" for r in decoded
    ]

def generate_retry_one(db_id, question, evidence, join_plan, bad_sql, error_msg):
    msgs = _build_retry_messages(db_id, question, evidence, join_plan, bad_sql, error_msg)
    prompt = gen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(gen_model.device)
    with torch.inference_mode():
        out = gen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=gen_tokenizer.pad_token_id)
    raw = gen_tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    del out, inputs; _free_cache()
    try:
        return _validate_sql(_clean_sql(raw))
    except Exception:
        return "SELECT 1"

print("SQL generator + retry generator ready.")

## 11. Reactive layer — column repair + DFC + alias norm + execute-and-retry

### 11.1 Schema-grounded column repair

sqlglot AST walk. For every `Table.col` reference (qualified by alias or table name), verify the column exists in the schema's table. If not:
- Try fuzzy match within the same table (Levenshtein ≤ 2 OR token-Jaccard ≥ 0.75).
- If still no match, search across other linked tables (only those in `join_plan.tables ∪ unresolved_tables`); rewrite the table qualifier if a confident match is found.
- If no confident match anywhere, leave the column alone (the model wrote something we can't repair — let execute-and-retry handle it).

Conservative thresholds prevent over-repair.

In [ ]:
import sqlglot
from sqlglot import exp as _exp

def _levenshtein(a, b, max_dist=3):
    """Standard Levenshtein with early-exit if distance exceeds max_dist."""
    if a == b: return 0
    if abs(len(a) - len(b)) > max_dist: return max_dist + 1
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        min_in_row = i
        for j, cb in enumerate(b, 1):
            ins = curr[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (0 if ca == cb else 1)
            v = min(ins, dele, sub)
            curr.append(v)
            if v < min_in_row: min_in_row = v
        if min_in_row > max_dist: return max_dist + 1
        prev = curr
    return prev[-1]

def _name_tokens(name):
    """Split name on underscore, space, and CamelCase boundaries; lowercase."""
    out = []
    for w in _re.findall(r"[A-Za-z0-9]+", name):
        for piece in _re.findall(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z]+|[A-Z]+|[0-9]+", w):
            if len(piece) >= 2: out.append(piece.lower())
    return out or [name.lower()]

def _token_jaccard(a, b):
    sa, sb = set(_name_tokens(a)), set(_name_tokens(b))
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

def _find_column_match(target_col, table, max_lev=2, min_jacc=0.75):
    """Search a Table for a column matching target_col. Returns column name or None."""
    target_lower = target_col.lower()
    # Pass 1: case-insensitive exact match
    for c in table.columns:
        if c.name.lower() == target_lower:
            return c.name
    # Pass 2: Levenshtein
    best = None; best_dist = max_lev + 1
    for c in table.columns:
        d = _levenshtein(target_lower, c.name.lower(), max_dist=max_lev)
        if d < best_dist:
            best_dist = d; best = c.name
    if best_dist <= max_lev:
        return best
    # Pass 3: token Jaccard
    best = None; best_score = 0.0
    for c in table.columns:
        s = _token_jaccard(target_col, c.name)
        if s > best_score:
            best_score = s; best = c.name
    if best_score >= min_jacc:
        return best
    return None


def column_repair(sql, db_id, join_plan):
    """AST repair: rewrite undefined column references using schema-grounded fuzzy match.

    Returns (new_sql, notes, n_rewrites). Idempotent on correct SQL.
    """
    schema = schema_from_bird(db_id)
    notes = []
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception as exc:
        return sql, [f"col-repair: parse failed ({exc}); pass-through"], 0

    # Build alias → table_name map for this query
    alias_to_table = {}
    for t in tree.find_all(_exp.Table):
        n = (t.name or "")
        a = (t.alias or t.name or "")
        if n and a:
            alias_to_table[a.lower()] = n

    linked_tables = set([*join_plan.tables, *join_plan.unresolved_tables])

    def _get_table_obj(name):
        return schema.tables.get(name)

    n_rewrites = 0
    for col in tree.find_all(_exp.Column):
        col_name = col.name or ""
        qualifier = (col.table or "").lower()
        if not col_name:
            continue

        # If the column is unqualified (no table prefix), skip — too risky to repair
        if not qualifier:
            continue

        # Resolve qualifier → actual table name
        ref_table_name = alias_to_table.get(qualifier, qualifier)
        ref_table = _get_table_obj(ref_table_name)

        if ref_table is None:
            continue  # unknown table; can't repair safely

        # Does the column already exist in the referenced table?
        if any(c.name.lower() == col_name.lower() for c in ref_table.columns):
            continue  # already valid

        # Try fuzzy match within same table
        match = _find_column_match(col_name, ref_table)
        if match:
            col.set("this", _exp.to_identifier(match, quoted=col.args.get("quoted", False)))
            n_rewrites += 1
            notes.append(f"col-repair: {ref_table_name}.{col_name} → {ref_table_name}.{match}")
            continue

        # Try across other linked tables
        cross_match = None  # (table_name, col_name)
        for tname in linked_tables:
            if tname == ref_table_name: continue
            other = _get_table_obj(tname)
            if other is None: continue
            m = _find_column_match(col_name, other)
            if m:
                cross_match = (tname, m)
                break  # take first match — order is graph-natural

        if cross_match:
            tname, mcol = cross_match
            # Rewrite column to use the new table name (not alias — model can re-alias if needed)
            col.set("table", _exp.to_identifier(tname))
            col.set("this", _exp.to_identifier(mcol, quoted=col.args.get("quoted", False)))
            n_rewrites += 1
            notes.append(f"col-repair: {ref_table_name}.{col_name} → {tname}.{mcol} (cross-table)")

    if n_rewrites == 0:
        return sql, [], 0
    return tree.sql(dialect="sqlite"), notes, n_rewrites


# Smoke test
_test_sql = "SELECT satscores.District FROM satscores WHERE satscores.AvgScrRead > 500"
class _DummyPlan:
    tables = ["satscores", "schools"]
    unresolved_tables = []
print("Smoke test column_repair:")
print(f"  in:  {_test_sql}")
out, notes, n = column_repair(_test_sql, "california_schools", _DummyPlan())
print(f"  out: {out}")
print(f"  notes: {notes}")
print(f"  n_rewrites: {n}")

## 11.2 DFC rewriter wiring + alias normalizer

In [ ]:
# Point the prototype's DFC adapter at the local sql_rewriter package
os.environ["DFC_SQL_REWRITER_PATH"] = str(REPO_DIR / "sql_rewriter")
import sql_rewriter as _dfc_pkg
print(f"DFC sql_rewriter loaded from: {_dfc_pkg.__file__}")

from text2sql_agent_prototype.prototype import DFCRewriterAdapter, SQLRewriter as _BaseSQLRewriter
from sql_rewriter import DFCPolicy, Resolution, SQLRewriter as DFCSQLRewriter

DFC_DUCKDB_MEMORY_LIMIT = "256MB"
DFC_DUCKDB_THREADS      = 1


def _configure_dfc_duckdb(dfc_rewriter):
    try:
        dfc_rewriter.conn.execute(f"PRAGMA memory_limit='{DFC_DUCKDB_MEMORY_LIMIT}'")
        dfc_rewriter.conn.execute(f"PRAGMA threads={DFC_DUCKDB_THREADS}")
    except Exception as exc:
        print(f"  warn: couldn't pin DuckDB resources: {exc}")


def _seed_dfc_schema(dfc_rewriter, schema):
    for tname, table in schema.tables.items():
        cols_def = ", ".join(f'"{c.name}" VARCHAR' for c in table.columns)
        try:
            dfc_rewriter.conn.execute(f'CREATE TABLE IF NOT EXISTS "{tname}" ({cols_def})')
        except Exception:
            pass


def _register_fk_policies(dfc_rewriter, schema):
    registered = 0
    for fk in schema.foreign_keys:
        try:
            dfc_rewriter.register_policy(DFCPolicy(
                sources=[fk.left_table, fk.right_table],
                constraint=(f'"{fk.left_table}"."{fk.left_column}" = '
                            f'"{fk.right_table}"."{fk.right_column}"'),
                on_fail=Resolution.REMOVE,
                description=f"FK: {fk.join_condition()}",
            ))
            registered += 1
        except Exception:
            pass
    return registered, len(schema.foreign_keys)


def normalize_aliases(sql):
    """sqlglot pass: rewrite column refs from real table names to declared aliases."""
    try:
        tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception as exc:
        return sql, [f"alias-norm: parse failed ({exc})"]
    table_to_alias = {}
    for t in tree.find_all(_exp.Table):
        if t.alias and t.name:
            table_to_alias[t.name.lower()] = t.alias
    if not table_to_alias:
        return sql, []
    n = 0
    for col in tree.find_all(_exp.Column):
        q = col.table
        if not q: continue
        new_alias = table_to_alias.get(q.lower())
        if new_alias and q != new_alias:
            col.set("table", _exp.to_identifier(new_alias))
            n += 1
    if n == 0:
        return sql, []
    return tree.sql(dialect="sqlite"), [f"alias-norm: rewrote {n} column ref(s)"]


# Per-db DFC rewriter cache (so we don't re-register policies per question)
_DFC_CACHE = {}

def get_dfc_for_db(db_id):
    if db_id in _DFC_CACHE:
        return _DFC_CACHE[db_id]
    schema = schema_from_bird(db_id)
    dfc_adapter = DFCRewriterAdapter()
    if not dfc_adapter.available:
        print(f"  warn: DFC unavailable for {db_id}: {dfc_adapter.error}")
        _DFC_CACHE[db_id] = (None, None)
        return None, None
    dfc = dfc_adapter._rewriter
    _configure_dfc_duckdb(dfc)
    _seed_dfc_schema(dfc, schema)
    reg, total = _register_fk_policies(dfc, schema)
    print(f"  [{db_id}] DFC FK policies: {reg}/{total} registered")
    base_rewriter = _BaseSQLRewriter(dfc_adapter=dfc_adapter)
    _DFC_CACHE[db_id] = (dfc_adapter, base_rewriter)
    return dfc_adapter, base_rewriter


def teardown_dfc(db_id):
    if db_id not in _DFC_CACHE: return
    adapter, _ = _DFC_CACHE[db_id]
    try:
        if adapter and adapter._rewriter and hasattr(adapter._rewriter, "conn"):
            adapter._rewriter.conn.close()
    except Exception:
        pass
    del _DFC_CACHE[db_id]


def dfc_and_alias_pass(sql, db_id, join_plan):
    """Run DFC + alias normalizer. Returns (new_sql, notes)."""
    notes = []
    _, base_rewriter = get_dfc_for_db(db_id)
    if base_rewriter is None:
        # DFC unavailable: just do alias norm
        new_sql, alias_notes = normalize_aliases(sql)
        return new_sql, [*alias_notes, "DFC unavailable"]
    rr = base_rewriter.rewrite(sql, join_plan)
    notes.extend(rr.notes)
    new_sql, alias_notes = normalize_aliases(rr.sql)
    notes.extend(alias_notes)
    return new_sql, notes

print("DFC + alias normalizer ready.")

## 11.3 Execute-and-retry

In [ ]:
import time, sqlite3 as _sqlite3

EXEC_TIMEOUT_S = 10

def try_execute(db_id, sql):
    """Execute SQL on the actual SQLite DB. Returns (rows_or_None, error_str_or_None)."""
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = _sqlite3.connect(str(db_path), timeout=EXEC_TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {EXEC_TIMEOUT_S * 1000}")
        deadline = time.monotonic() + EXEC_TIMEOUT_S
        conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
        rows = conn.execute(sql).fetchall()
        return rows, None
    except Exception as exc:
        return None, str(exc)
    finally:
        try: conn.set_progress_handler(None, 0)
        except Exception: pass
        conn.close()


def reactive_layer(sql_v0, db_id, question, evidence, join_plan):
    """Full reactive pipeline:
        1. column repair
        2. DFC + alias norm
        3. execute test
        4. retry on error: regenerate via Qwen-32B with error context
        5. re-repair retry SQL
        6. re-execute; if still errors, fall back to v3
    Returns (final_sql, telemetry_dict).
    """
    tel = {
        "col_repair_n": 0, "col_repair_notes": [],
        "dfc_notes": [],
        "first_error": None, "retried": False,
        "retry_error": None, "retry_used": False,
    }

    # 1. Column repair on v0
    sql_v1, cr_notes, n_cr = column_repair(sql_v0, db_id, join_plan)
    tel["col_repair_n"] = n_cr
    tel["col_repair_notes"] = cr_notes

    # 2. DFC + alias norm
    sql_v3, dfc_notes = dfc_and_alias_pass(sql_v1, db_id, join_plan)
    tel["dfc_notes"] = dfc_notes

    # 3. Execute test
    _, err = try_execute(db_id, sql_v3)
    if err is None:
        return sql_v3, tel

    # 4. Retry with error context
    tel["first_error"] = err
    tel["retried"] = True
    sql_retry = generate_retry_one(db_id, question, evidence, join_plan, sql_v3, err)

    # 5. Re-repair retry SQL
    sql_retry_v1, _, _ = column_repair(sql_retry, db_id, join_plan)
    sql_retry_v2, _ = dfc_and_alias_pass(sql_retry_v1, db_id, join_plan)

    # 6. Re-execute; use retry if it works, else fall back to v3
    _, retry_err = try_execute(db_id, sql_retry_v2)
    if retry_err is None:
        tel["retry_used"] = True
        return sql_retry_v2, tel
    tel["retry_error"] = retry_err
    return sql_v3, tel  # both errored; use first attempt

print("reactive_layer ready.")

## 12. Smoke test — qid=15 (the column-hallucination case from graph.md)

In [ ]:
q0 = questions[0]

# Q0 sanity
exp_q0 = expand_query_batch([(q0["question"], q0.get("evidence", "") or "")])[0]
plan_q0, dbg_q0 = retrieve_then_graph(q0["db_id"], q0["question"], q0.get("evidence", "") or "", exp_q0)
sql_v0 = generate_sql_one(q0["db_id"], q0["question"], q0.get("evidence", "") or "", plan_q0)
final, tel = reactive_layer(sql_v0, q0["db_id"], q0["question"], q0.get("evidence", "") or "", plan_q0)
print(f"=== Q0 (db={q0['db_id']}) ===")
print(f"v0 (model):  {sql_v0}")
print(f"final:       {final}")
print(f"col-repair n: {tel['col_repair_n']}  notes: {tel['col_repair_notes']}")
print(f"dfc notes:   {tel['dfc_notes']}")
print(f"retried:     {tel['retried']}  retry_used: {tel['retry_used']}")
if tel["first_error"]:
    print(f"first_error: {tel['first_error'][:200]}")

# Now try the actual hallucination case from graph.md — find qid=15
q15 = next((q for i, q in enumerate(questions) if str(q.get("question_id", i)) == "15"), None)
if q15:
    print(f"\n=== Q15 (the District-hallucination case from graph.md) ===")
    print(f"Q: {q15['question']}")
    exp15 = expand_query_batch([(q15["question"], q15.get("evidence", "") or "")])[0]
    plan15, _ = retrieve_then_graph(q15["db_id"], q15["question"], q15.get("evidence", "") or "", exp15)
    sql_v0_15 = generate_sql_one(q15["db_id"], q15["question"], q15.get("evidence", "") or "", plan15)
    final15, tel15 = reactive_layer(sql_v0_15, q15["db_id"], q15["question"], q15.get("evidence", "") or "", plan15)
    print(f"v0 (model):  {sql_v0_15}")
    print(f"final:       {final15}")
    print(f"col-repair:  {tel15['col_repair_n']} rewrites — {tel15['col_repair_notes']}")
    print(f"retried:     {tel15['retried']}  retry_used: {tel15['retry_used']}")
    if tel15["first_error"]:
        print(f"first_error: {tel15['first_error'][:200]}")
    print(f"gold:        {q15['SQL']}")

## 13. Run on full BIRD dev → `predict_dev_rewriter.json`

In [ ]:
from tqdm.auto import tqdm
import gc, psutil, json

N            = len(questions)
GEN_BATCH    = int(os.environ.get("REWRITER_GEN_BATCH", "8"))
EXP_BATCH    = int(os.environ.get("REWRITER_EXP_BATCH", "16"))
PRED_PATH    = pathlib.Path("/content/predict_dev_rewriter.json")
LOG_PATH     = pathlib.Path("/content/rewriter_log.jsonl")

predictions = {}
if PRED_PATH.exists():
    predictions = json.loads(PRED_PATH.read_text())
    print(f"Resuming with {len(predictions)} predictions")
if not predictions and LOG_PATH.exists():
    LOG_PATH.unlink()

def _mem_str():
    rss = psutil.Process(os.getpid()).memory_info().rss / 1e9
    avail = psutil.virtual_memory().available / 1e9
    gpu = ""
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        gpu = f"  GPU free={free/1e9:.1f}GB"
    return f"RSS={rss:.1f}GB  free={avail:.1f}GB{gpu}"

def write_preds(p, d):
    with open(p, "w", encoding="utf-8") as f: json.dump(d, f)

def append_log(p, lines):
    if not lines: return
    with open(p, "a", encoding="utf-8") as f:
        for r in lines: f.write(json.dumps(r) + "\n")

groups = defaultdict(list)
for i, q in enumerate(questions[:N]):
    groups[q["db_id"]].append((i, q))

print(f"BIRD dev: {N} questions across {len(groups)} dbs")
print(f"Memory at start: {_mem_str()}")

total_done = len(predictions)
pbar = tqdm(total=N, initial=total_done, desc="BIRD dev (rewriter)")
last_checkpoint = total_done

# Aggregate counters
total_col_rewrites = 0
total_retries = 0
total_retry_success = 0

for db_id, qlist in groups.items():
    qlist_remaining = [(i, q) for (i, q) in qlist if str(q.get("question_id", i)) not in predictions]
    if not qlist_remaining:
        continue

    retr = get_retriever(db_id)
    graph = get_graph(db_id)
    # Pre-warm DFC for this db (registers FK policies once)
    get_dfc_for_db(db_id)
    print(f"  [{db_id}] {len(qlist_remaining)} questions, "
          f"corpus={len(retr.docs)} cols, fk_edges={sum(len(v) for v in graph.adjacent.values())//2}  {_mem_str()}")

    # Pass 1: expansion
    exp_inputs = [(q["question"], q.get("evidence", "") or "") for (_i, q) in qlist_remaining]
    expansions = expand_query_batch(exp_inputs, batch_size=EXP_BATCH)

    # Pass 2: retrieve + graph
    join_plans = []
    log_buf = []
    for (orig_i, q), exp in zip(qlist_remaining, expansions):
        qid = str(q.get("question_id", orig_i))
        plan, dbg = retrieve_then_graph(db_id, q["question"], q.get("evidence", "") or "", exp)
        join_plans.append(plan)
        log_buf.append({"qid": qid, "db_id": db_id, **dbg})

    # Pass 3: SQL generation in batches → reactive layer per question
    pending = list(zip(qlist_remaining, join_plans))
    for start in range(0, len(pending), GEN_BATCH):
        chunk = pending[start:start + GEN_BATCH]
        items = [(db_id, q["question"], q.get("evidence", "") or "", jp)
                 for ((_i, q), jp) in chunk]
        sqls_v0 = generate_sql_batch(items)

        # Reactive layer: per-question (column repair + DFC + alias + execute-and-retry)
        for ((orig_i, q), jp), sql_v0, log_entry in zip(chunk, sqls_v0, log_buf[start:start + GEN_BATCH]):
            qid = str(q.get("question_id", orig_i))
            try:
                final_sql, tel = reactive_layer(
                    sql_v0, db_id, q["question"], q.get("evidence", "") or "", jp,
                )
            except Exception as exc:
                print(f"  [{qid}] reactive layer failed: {type(exc).__name__}: {str(exc)[:120]}")
                final_sql = sql_v0; tel = {"reactive_error": str(exc)}
            sql_clean = (final_sql or "SELECT 1").replace("\n", " ").strip()
            predictions[qid] = f"{sql_clean}\t----- bird -----\t{db_id}"
            log_entry["sql_v0"] = sql_v0
            log_entry["sql_final"] = sql_clean
            log_entry["telemetry"] = tel
            total_col_rewrites += tel.get("col_repair_n", 0)
            if tel.get("retried"): total_retries += 1
            if tel.get("retry_used"): total_retry_success += 1
            pbar.update(1)
        gc.collect()
        try: torch.cuda.empty_cache()
        except Exception: pass
        if len(predictions) - last_checkpoint >= 50:
            write_preds(PRED_PATH, predictions)
            last_checkpoint = len(predictions)

    append_log(LOG_PATH, log_buf)
    write_preds(PRED_PATH, predictions)

    # Tear down DFC for this db (frees DuckDB conn)
    teardown_dfc(db_id)
    print(f"  [{db_id}] done. col_rewrites_so_far={total_col_rewrites}  "
          f"retries={total_retries} ({total_retry_success} succeeded).  {_mem_str()}")

pbar.close()
write_preds(PRED_PATH, predictions)
print(f"\nWrote {PRED_PATH} ({len(predictions)} predictions)")
print(f"Wrote {LOG_PATH}")
print(f"\n=== Reactive layer aggregate stats ===")
print(f"  Total column rewrites: {total_col_rewrites}")
print(f"  Total retries:         {total_retries}")
print(f"  Retries that succeeded: {total_retry_success} ({100*total_retry_success/max(total_retries,1):.1f}%)")
print(f"Memory at end: {_mem_str()}")

## 14. Local metrics — Join Acc + Exec Acc + Rewriter telemetry

In [ ]:
import sqlite3, json, time
import sqlglot
from sqlglot import exp as _exp_metric

PRED_PATH = pathlib.Path("/content/predict_dev_rewriter.json")
LOG_PATH  = pathlib.Path("/content/rewriter_log.jsonl")
predictions = json.loads(PRED_PATH.read_text())
qmap = {str(q.get("question_id", i)): q for i, q in enumerate(questions)}
rewrite_log = {}
if LOG_PATH.exists():
    for line in LOG_PATH.read_text().splitlines():
        if line.strip():
            try:
                obj = json.loads(line); rewrite_log[obj["qid"]] = obj
            except Exception: pass

TIMEOUT_S = 10
_NULL = "\x00NULL\x00"

def run_sql(db_id, sql):
    db_path = BIRD_DBS / db_id / f"{db_id}.sqlite"
    conn = sqlite3.connect(str(db_path), timeout=TIMEOUT_S)
    try:
        conn.execute(f"PRAGMA busy_timeout = {TIMEOUT_S * 1000}")
        deadline = time.monotonic() + TIMEOUT_S
        conn.set_progress_handler(lambda: 1 if time.monotonic() > deadline else 0, 10_000)
        return conn.execute(sql).fetchall(), None
    except Exception as exc:
        return None, str(exc)
    finally:
        try: conn.set_progress_handler(None, 0)
        except Exception: pass
        conn.close()

def normalize(rows):
    if rows is None: return None
    return sorted(tuple(_NULL if c is None else str(c) for c in r) for r in rows)

def extract_joins(sql):
    try: tree = sqlglot.parse_one(sql, dialect="sqlite")
    except Exception: return set()
    alias_map = {}
    for t in tree.find_all(_exp_metric.Table):
        n = (t.name or "").lower(); a = (t.alias or t.name or "").lower()
        if n and a: alias_map[a] = n
    joins = set()
    for eq in tree.find_all(_exp_metric.EQ):
        l, r = eq.this, eq.expression
        if isinstance(l, _exp_metric.Column) and isinstance(r, _exp_metric.Column):
            lt = (l.table or "").lower(); rt = (r.table or "").lower()
            if lt and rt and lt in alias_map and rt in alias_map and lt != rt:
                joins.add(frozenset([f"{alias_map[lt]}.{l.name.lower()}",
                                     f"{alias_map[rt]}.{r.name.lower()}"]))
    return joins

def join_f1(pred_sql, gold_sql):
    gold = extract_joins(gold_sql)
    if not gold: return None
    pred = extract_joins(pred_sql)
    if not pred: return 0.0
    tp = len(pred & gold)
    if tp == 0: return 0.0
    p = tp / len(pred); r = tp / len(gold)
    return 2 * p * r / (p + r)

results = []
for i, q in enumerate(questions):
    qid = str(q.get("question_id", i))
    line = predictions.get(qid)
    if line is None:
        pred_sql = "SELECT 1"; missing = True
    else:
        pred_sql = line.split("\t----- bird -----\t", 1)[0].strip(); missing = False
    is_fallback = pred_sql.rstrip(";").strip().upper() == "SELECT 1"

    pred_rows, pred_err = run_sql(q["db_id"], pred_sql)
    gold_rows, gold_err = run_sql(q["db_id"], q["SQL"])

    if missing: ex = "missing_prediction"
    elif pred_err: ex = "exec_error"
    elif gold_err: ex = "gold_error"
    elif normalize(pred_rows) == normalize(gold_rows): ex = "correct"
    else: ex = "wrong_result"

    log = rewrite_log.get(qid, {})
    tel = log.get("telemetry", {})
    results.append({
        "qid": qid, "db_id": q["db_id"], "difficulty": q.get("difficulty", "?"),
        "ex": ex, "pred_err": pred_err, "fallback": is_fallback, "missing": missing,
        "jf1": join_f1(pred_sql, q["SQL"]), "pred_sql": pred_sql,
        "col_repair_n": tel.get("col_repair_n", 0),
        "retried": tel.get("retried", False),
        "retry_used": tel.get("retry_used", False),
        "first_error": tel.get("first_error"),
        "log": log,
    })

total = len(results)
ex_correct = sum(r["ex"] == "correct" for r in results)
ex_acc = ex_correct / max(total, 1) * 100
scored_join = [r for r in results if r["jf1"] is not None]
join_acc = (sum(r["jf1"] for r in scored_join) / max(len(scored_join), 1)) * 100

# Telemetry aggregates
n_col_rewrites = sum(r["col_repair_n"] for r in results)
n_retried = sum(1 for r in results if r["retried"])
n_retry_used = sum(1 for r in results if r["retry_used"])
correct_among_retried = sum(1 for r in results if r["retried"] and r["ex"] == "correct")
correct_among_retry_used = sum(1 for r in results if r["retry_used"] and r["ex"] == "correct")

print(f"=== Rewriter ablation (Graph + Column Repair + DFC + Alias Norm + Execute-and-Retry) — {total} predictions ===\n")
print(f"{'Method':<40s}{'Join Acc.':>12s}{'Exec. Acc.':>13s}")
print(f"{'-'*65}")
print(f"{'Baseline (Qwen-32B, full)':<40s}{66.3:>12.1f}{54.2:>13.1f}")
print(f"{'Retrieval (N=5)':<40s}{66.7:>12.1f}{52.1:>13.1f}")
print(f"{'+ Graph (N=8)':<40s}{71.9:>12.1f}{53.3:>13.1f}")
print(f"{'+ Rewriter (this run)':<40s}{join_acc:>12.1f}{ex_acc:>13.1f}")

print(f"\nReactive-layer telemetry:")
print(f"  Column rewrites total: {n_col_rewrites}  (across {sum(1 for r in results if r['col_repair_n']>0)} questions)")
print(f"  Retries triggered:     {n_retried}  ({100*n_retried/max(total,1):.1f}% of predictions)")
print(f"  Retries that fixed exec: {n_retry_used}  ({100*n_retry_used/max(n_retried,1):.1f}% of retries)")
print(f"  EX-correct among retried questions:    {correct_among_retried}/{n_retried}")
print(f"  EX-correct among retry-fixed questions: {correct_among_retry_used}/{n_retry_used}")

print(f"\nEX status:  correct={ex_correct}  "
      f"wrong={sum(r['ex']=='wrong_result' for r in results)}  "
      f"err={sum(r['ex']=='exec_error' for r in results)}  "
      f"missing={sum(r['ex']=='missing_prediction' for r in results)}  "
      f"fallback={sum(r['fallback'] for r in results)}")

print(f"\n--- By difficulty ---")
print(f"{'difficulty':<14s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'cases':>8s}")
for d in ("simple", "moderate", "challenging", "?"):
    bucket = [r for r in results if r["difficulty"] == d]
    if not bucket: continue
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    print(f"  {d:<12s}{j:>11.1f}{e:>12.1f}{len(bucket):>8d}")

print(f"\n--- By db_id ---")
print(f"{'db_id':<26s}{'Join Acc.':>11s}{'Exec. Acc.':>12s}{'retries':>9s}{'cases':>8s}")
by_db = defaultdict(list)
for r in results:
    by_db[r["db_id"]].append(r)
for d, bucket in sorted(by_db.items()):
    bj = [r for r in bucket if r["jf1"] is not None]
    j = (sum(r["jf1"] for r in bj) / max(len(bj), 1)) * 100 if bj else float("nan")
    e = sum(r["ex"] == "correct" for r in bucket) / len(bucket) * 100
    nr = sum(1 for r in bucket if r["retried"])
    print(f"  {d:<24s}{j:>11.1f}{e:>12.1f}{nr:>9d}{len(bucket):>8d}")

print("\n--- First 5 EX failures (rewriter diagnosis) ---")
shown = 0
for r in results:
    if r["ex"] == "correct" or shown >= 5:
        continue
    q = qmap[r["qid"]]
    jf = "—" if r["jf1"] is None else f"{r['jf1']:.2f}"
    log = r["log"]
    g_tab = log.get("graph_tables", [])
    print(f"\n  qid={r['qid']} ({r['difficulty']}) db={r['db_id']}  ex={r['ex']}  jf1={jf}  retried={r['retried']} retry_used={r['retry_used']}")
    print(f"    Q:    {q['question'][:140]}")
    if q.get("evidence"):
        print(f"    ev:   {q['evidence'][:140]}")
    print(f"    graph:{', '.join(g_tab) or '(none)'}")
    if r["col_repair_n"]:
        print(f"    col-repair: {r['col_repair_n']} rewrites")
    if r["first_error"]:
        print(f"    first_err: {r['first_error'][:200]}")
    print(f"    pred: {r['pred_sql'][:180]}")
    print(f"    gold: {q['SQL'][:180]}")
    if r["pred_err"]:
        print(f"    pred_err: {r['pred_err'][:160]}")
    shown += 1

## 15. Run BIRD's official evaluation scripts

In [ ]:
BIRD_EVAL = pathlib.Path("/content/bird_eval")
BIRD_EVAL.mkdir(exist_ok=True)
GT_PATH = BIRD_EVAL / "dev_gold.sql"
lines = []
for q in questions:
    sql = q["SQL"].replace("\n", " ").strip()
    lines.append(f"{sql}\t{q['db_id']}")
GT_PATH.write_text("\n".join(lines))
print("Wrote", GT_PATH)

ex_script  = BIRD_EVAL / "evaluation.py"
ves_script = BIRD_EVAL / "evaluation_ves.py"

if ex_script.exists():
    !python {ex_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation.py at {ex_script} and re-run.")

if ves_script.exists():
    !python {ves_script} \
        --predicted_sql_path {PRED_PATH} \
        --ground_truth_path {GT_PATH} \
        --db_root_path {BIRD_DBS} \
        --num_cpus 4 \
        --meta_time_out 30.0 \
        --mode_gt gt --mode_predict gpt
else:
    print(f"Place evaluation_ves.py at {ves_script} and re-run.")

## Notes

**Component contribution:** Δ EX between this run and the graph-only run quantifies what the reactive layer adds:
- Column repair catches column hallucinations before execution
- DFC + alias norm enforces FK-valid joins / consistent aliasing
- Execute-and-retry gives the model a second chance with explicit error context for the ~89 errored queries from graph

**Knobs:**
- `EXEC_TIMEOUT_S` (cell 23) — per-query timeout (default 10s).
- Column-repair thresholds (cell 21): `_levenshtein` `max_dist=2`, `_token_jaccard` `min_jacc=0.75`. Loosening risks over-repair.
- Retry budget: 1 per errored query. To allow more retries, wrap the retry block in a loop.
- DFC DuckDB cap: `DFC_DUCKDB_MEMORY_LIMIT="256MB"` per instance (cell 22).

**Cost.** Retry adds ~1 extra Qwen-32B call per ~6% of questions (the error rate from graph). Total wall-clock ~5% longer than graph alone.